# Storytelling with the Manim Library   
The goal of this workbook is to create a video that tells the story of the evolution of the horse population. I will use the Python library Manim for visualization and animation. The video will be divided into several scenes, each created with separate code. I will then combine the individual scenes into a single video using DaVinci Resolve.


**1) Installing Manim in Google Colab**

In [ ]:
!sudo apt update
!sudo apt install -y libcairo2-dev libpango1.0-dev ffmpeg \
    texlive texlive-latex-extra texlive-fonts-extra \
    texlive-latex-recommended texlive-science \

!pip install manim

**2) Required libraries**

In [ ]:
from manim import *
import json
import math

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**3) Data upload**

In [ ]:
DATA_FILE_PATH = "/content/drive/MyDrive/01_data_science/03_projects/01_horses_population/horse_population_analysis/data/clean_data.json"

In [ ]:
HORSE_ICON_PATH = "/content/drive/MyDrive/01_data_science/03_projects/01_horses_population/horse_population_analysis/docs/icons/horse_icon.png"

In [ ]:
MANIM_ICON_PATH = "/content/drive/MyDrive/01_data_science/03_projects/01_horses_population/horse_population_analysis/docs/icons/manim_logo.png"

In [ ]:
with open(DATA_FILE_PATH, "r") as f:
    horse_data = json.load(f)

In [ ]:
# extract years and horse counts - to 2025
data_to_2025 = [record for record in horse_data if record["Year"] <= 2025]
years_to_2025 = [data["Year"] for data in data_to_2025]
counts_to_2025 = [data["Horses"] for data in data_to_2025]

In [ ]:
# Extract years and horse counts - from 2025 (use 2025 for a smooth transition)
data_from_2025 = [record for record in horse_data if record["Year"] >= 2025]
years_from_2025 = [data["Year"] for data in data_from_2025]
counts_from_2025 = [data["Horses"] for data in data_from_2025]

In [ ]:
# extract years and horse counts - data for uncertainty intervals - from 2026 onwards
data_from_2026 = [record for record in horse_data if record["Year"] >= 2026]
years_from_2026 = [data["Year"] for data in data_from_2026]
counts_lower_2026 = [data["Horses_lower"] for data in data_from_2026]
counts_upper_2026 = [data["Horses_upper"] for data in data_from_2026]

In [ ]:
# data for a more detailed view of the chart (between 2000 and 2025)
years_closer_look = [data["Year"] for data in horse_data if 2003 <= data["Year"] <= 2025]
counts_closer_look = [data["Horses"] for data in horse_data if 2003 <= data["Year"] <= 2025]

# First Scene - Introduction

In [ ]:
%%manim -qh -v WARNING HorsePopulationIntro

class HorsePopulationIntro(Scene):
    def construct(self):
        # title text
        text_line_1 = Text("Vývoj populace koní", weight=BOLD, font_size=48)
        text_line_2 = Text("na českém území", weight=BOLD, font_size=48)
        text_line_3 = Text("v období přesahujícím 100 let", weight=BOLD, font_size=48)
        text_line_4 = Text("1921 - 2025", weight=BOLD, font_size=48)

        # grouping texts into one object
        title = VGroup(text_line_1, text_line_2, text_line_3, text_line_4).arrange(
            DOWN,
            center=True,
            buff=0.3
        )

        # text positioning
        title.to_edge(UP, buff=1.0)

        # horse icon
        horse_icon = ImageMobject(HORSE_ICON_PATH).scale(0.4)

        # arrange icon
        horse_icon.next_to(title, DOWN, buff=0.5)

        # animations - text
        self.play(Write(title))

        # animation - horse icon
        self.play(FadeIn(horse_icon))
        self.wait(2.0)

        # preparation for the scene transition
        self.play(FadeOut(title, horse_icon))
        self.wait(0.5)

# Second Scene - Plotting the Chart

In [ ]:
def create_horse_graph_elements(
    # Define the step sizes for the X (years) and Y (horse count) axes.
    x_step: int = 20,
    y_step: int = 100000,
    # Set the starting and ending values for the X and Y axes.
    x_start: int = 1920,
    y_start: int = 0,
    x_end: int = 2030,
    y_end: int = 600000,
    # Data for graph line.
    x_values: list = years_to_2025,
    y_values: list = counts_to_2025,
    # Visual properties of the graph line.
    line_color: str = BLUE_C,
    stroke_width: int = 10,
):
    """
    Creates and returns the Manim Axes object and a horse population graph line based on provided parameters.
    This function centralizes axis and graph creation logic for reusability and flexibility across scenes,
    allowing for both overview and detailed 'zoom-in' graph views.
    """
    # Create the Manim Axes object.
    axes = Axes(
        x_range=[x_start, x_end, x_step],
        y_range=[y_start, y_end, y_step],
        x_length=11.5,
        y_length=6.5,
        axis_config={"color": WHITE, "include_tip": False},
    ).to_edge(DOWN, buff=0.8)

    # Format numbers on the Y-axis for better readability.
    y_labels_dict = {val: "0" if val == 0 else f"{int(val/1000)}k" for val in range(0, int(y_end) + 1, y_step)}
    axes.y_axis.add_labels(y_labels_dict)

    # Format year labels on the X-axis.
    x_labels_dict = {val: f"{val}" for val in range(x_start, x_end + 1, x_step)}
    axes.x_axis.add_labels(x_labels_dict)

    # Create the complete graph line using the historical data.
    graph_line = axes.plot_line_graph(
        x_values=x_values,
        y_values=y_values,
        line_color=line_color,
        stroke_width=stroke_width
    )["line_graph"] # Access the Mobject representing the line graph.

    return axes, graph_line

In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph

class HorsePopulationGraph(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION
        # Create and retrieve the axes and historical graph line using a helper function.
        axes, graph_line = create_horse_graph_elements()

        # Animate the drawing of the axes onto the scene.
        self.play(Write(axes))

        # COUNTER PREPARATION
        # ValueTracker - Mobject for animating numerical values.
        progress = ValueTracker(0)

        # VMobject - dynamically update to show only a portion of'graph_line.
        drawn_graph = VMobject()

        # Labels for the counter display.
        data_labels = VGroup(
            Text("Rok:", font_size=40, color=WHITE),
            Text("Počet koní:", font_size=40, color=WHITE)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.5)

        # DecimalNumber objects to display the animated year and horse count.
        year_value = DecimalNumber(min(years_to_2025), num_decimal_places=0, font_size=64, group_with_commas=False)
        count_value = DecimalNumber(min(counts_to_2025), num_decimal_places=0, font_size=64)

        # Grouping value display and overall counter placement.
        value_group = VGroup(year_value, count_value).arrange(DOWN, aligned_edge=LEFT, buff=0.5)

        # Combines labels and values horizontally, then positions them.
        data_display = VGroup(data_labels, value_group).arrange(RIGHT, buff=0.2)

        # Positions the entire data display in the top-right corner.
        data_display.to_corner(UP + RIGHT, buff=0.5).shift(DOWN * 1.0 + LEFT * 1.0)

        # UPDATERS FOR DYNAMIC ANIMATION
        # Updater for the plotted graph: dynamically updates 'drawn_graph' to show a sub-portion
        # of 'graph_line' based on the 'progress' ValueTracker's value.
        drawn_graph.add_updater(lambda mob: mob.become(graph_line.get_subcurve(0, progress.get_value())))

        # Updater function for the data counter:
        # interpolates year and count values based on the animation 'alpha' (progress value).
        # This ensures smooth transitions of numbers as the graph is drawn.
        def update_data_values(mob):
            alpha = progress.get_value()          # current animation progress (0 to 1)
            total_points = len(data_to_2025) - 1  # total segments to interpolate over

            # Determine the current segment for interpolation.
            start_index = min(math.floor(alpha * total_points), total_points - 1)
            end_index = min(start_index + 1, total_points)
            local_alpha = (alpha * total_points) - start_index

            # Perform linear interpolation for year and count values.
            current_year = interpolate(years_to_2025[start_index], years_to_2025[end_index], local_alpha)
            current_count = interpolate(counts_to_2025[start_index], counts_to_2025[end_index], local_alpha)

            # Uppdate the DecimalNumber Mobjects with the interpolated values.
            year_value.set_value(current_year)
            count_value.set_value(current_count)

        # Attach the updater function to the 'data_display' group.
        data_display.add_updater(update_data_values)

        # SCENE ANIMATION
        # Animation - data labels.
        self.play(Write(data_labels))
        self.wait(0.5)

        # Animation - counter.
        self.add(data_display)
        self.add(drawn_graph)

        # Animation - graph.
        self.play(
            progress.animate.set_value(1),
            run_time=15,
            rate_func=linear
        )
        self.wait(2)

        # Preparation for the scene transition.
        self.play(FadeOut(data_display))
        self.wait(0.5)

# Third Scene - Highlighting of Peaks



In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph_II

class HorsePopulationGraph_II(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION
        # Create and retrieve the axes and historical graph line using a helper function.
        axes, graph_line = create_horse_graph_elements()

        # Visualize the static charte.
        self.add(axes, graph_line)
        self.wait(0.5)

        # HIGHLIGHTING CHART PEAKS
        # Identify the y-values for the population peaks in 1927 and 1946.
        y_peak_1 = counts_to_2025[years_to_2025.index(1927)]
        y_peak_2 = counts_to_2025[years_to_2025.index(1946)]

        # Convert data coordinates (year, count) to scene coordinates for precise placement of Mobjects.
        peak_1_point = axes.c2p(1927, y_peak_1)
        peak_2_point = axes.c2p(1946, y_peak_2)

        # Define points on the x-axis for the base of the vertical highlight lines.
        x_point_1 = axes.c2p(1927, 0)
        x_point_2 = axes.c2p(1946, 0)

        # Create dashed vertical lines and their corresponding year labels for both peaks.
        v_line_1 = DashedLine(peak_1_point, x_point_1, color=YELLOW, stroke_width=6)
        label_1 = Text("1927", weight=BOLD, color=YELLOW, font_size=30).next_to(v_line_1, UP, buff=0.3)

        v_line_2 = DashedLine(peak_2_point, x_point_2, color=YELLOW, stroke_width=6)
        label_2 = Text("1946", weight=BOLD, color=YELLOW, font_size=30).next_to(v_line_2, UP, buff=0.3)

        # Animate the display of the first peak's highlight line and label.
        self.play(
            Create(v_line_1),
            FadeIn(label_1),
            run_time=1.0
        )
        self.wait(0.3)

        # Animate the display of the second peak's highlight line and label.
        self.play(
            Create(v_line_2),
            FadeIn(label_2),
            run_time=1.0
        )
        self.wait(1)

        # Creation of the scene text - first part.
        text_line_1 = Text("V letech 1927 a 1946 byla", font_size=42)
        text_line_2 = Text("populace koní největší.", font_size=42)

        # Grouping texts into one object - first part.
        summary_text = VGroup(text_line_1, text_line_2).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )

        # Positioning the entire text block in the upper left corner - first part.
        summary_text.to_corner(UP + RIGHT, buff=0.8)

        # Text display - first part.
        self.play(Write(summary_text))
        self.wait(2)

        # Creation of the scene text - second part.
        text_line_3 = Text("Velikost populace byla:", font_size=42)
        text_line_4 = Text("roku 1927 - 456 tis. koní;", font_size=42)
        text_line_5 = Text("roku 1946 - 450 tis. koní.", font_size=42)

        # Grouping texts into one object - second part.
        summary_text_2 = VGroup(text_line_3, text_line_4, text_line_5).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )

        # Positioning the entire text block in the upper left corner - second part.
        summary_text_2.to_corner(UP + RIGHT, buff=0.8)

        # Text transform - second part.
        self.play(Transform(summary_text, summary_text_2))
        self.wait(3.5)

        # Preparation for the scene transition.
        self.play(FadeOut(
            label_1,
            v_line_1,
            label_2,
            v_line_2,
            summary_text)
        )
        self.wait(0.5)

# Fourth Scene - Population Decline



In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph_III

class HorsePopulationGraph_III(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION
        # Create and retrieve the axes and historical graph line using a helper function.
        axes, graph_line = create_horse_graph_elements()

        # Visualize the static charte.
        self.add(axes, graph_line)

        # HIGHLIGHTING DECLINE PERIOD
        # Identify y-values for the start and end of the significant population decline.
        # Using 1946 for correct display.
        y_count_start_decline = counts_to_2025[years_to_2025.index(1946)]
        y_count_end_decline = counts_to_2025[years_to_2025.index(1995)]

        # Convert data coordinates (year, count) to scene coordinates for precise Mobject placement.
        point_start_decline = axes.c2p(1946, y_count_start_decline)
        point_end_decline = axes.c2p(1995, y_count_end_decline)

        # Define points on the x-axis for the base of the vertical highlight lines.
        x_point_start_decline = axes.c2p(1946, 0)
        x_point_end_decline = axes.c2p(1995, 0)

        # Create dashed vertical lines and their corresponding year labels for both points.
        v_line_start = DashedLine(point_start_decline, x_point_start_decline, color=RED, stroke_width=6)
        label_start = Text("1947", color=RED, weight=BOLD, font_size=30).next_to(v_line_start, UP, buff=0.3)

        v_line_end = DashedLine(point_end_decline, x_point_end_decline, color=RED, stroke_width=6)
        label_end = Text("1995", color=RED, weight=BOLD, font_size=30).next_to(v_line_end, UP, buff=0.3)

        # Define the data range for the decline period for highlighting the line segment.
        start_idx = years_to_2025.index(1946)
        end_idx = years_to_2025.index(1995)
        decline_years = years_to_2025[start_idx:end_idx + 1]
        decline_counts = counts_to_2025[start_idx:end_idx + 1]

        # Create a thicker, red line to highlight the period of decline on the chart.
        decline_line = axes.plot_line_graph(
            x_values=decline_years,
            y_values=decline_counts,
            line_color=RED,
            stroke_width=16
        )["line_graph"]

        # Animate the display of the first decline point's highlight (line and label)
        self.play(
            Create(v_line_start),
            FadeIn(label_start),
            run_time=1.0
            )

        # Animate the display of the second decline point's highlight.
        self.play(
            Create(v_line_end),
            FadeIn(label_end),
            run_time=1.0
            )

        # Animate the appearance of the thick red decline line.
        self.play(
            FadeIn(decline_line),
            run_time=1.0
            )
        self.wait(1.0)

        # Creation of the scene text - first part.
        text_1 = Text("V roce 1947 začíná pokles", font_size=42)
        text_2 = Text("trvající 48 let.", font_size=42)

        summary_text = VGroup(text_1, text_2).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text.to_corner(UP + RIGHT, buff=0.8)

        self.play(Write(summary_text))
        self.wait(2)

        # Creation of the scene text - second part.
        text_3 = Text("V roce 1995 dosáhla", font_size=42)
        text_4 = Text("populace svého minima:", font_size=42)
        text_5 = Text("pouze 18 tis. koní.", font_size=42)

        summary_text_2 = VGroup(text_3, text_4, text_5).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_2.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text, summary_text_2))
        self.wait(3.5)

        # Creation of the scene text - third part.
        text_6 = Text("Během 48 let se počet", font_size=42)
        text_7 = Text("koní snížil o 432 tis.,", font_size=42)
        text_8 = Text("což je pokles o 96 %.", font_size=42)

        summary_text_3 = VGroup(text_6, text_7, text_8).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_3.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text, summary_text_3))
        self.wait(3.5)

        # Preparation for the scene transition.
        self.play(FadeOut(
            summary_text,
            decline_line,
            label_start,
            label_end,
            v_line_start,
            v_line_end
            )
        )

        self.wait(0.5)

# Fifth Scene - Population Growth

In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph_IV

class HorsePopulationGraph_IV(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION
        # Create and retrieve the axes and historical graph line using a helper function.
        axes, graph_line = create_horse_graph_elements()

        # Visualize the static charte.
        self.add(axes, graph_line)

        # HIGHLIGHTING BREAKING POINT (START OF INCREASE)
        # Identify the y-value for the population (1996).
        y_count_1996 = counts_to_2025[years_to_2025.index(1996)]

        # Convert data coordinates (year, count) to scene coordinates for precise Mobject placement.
        point_1996 = axes.c2p(1996, y_count_1996)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_1996 = axes.c2p(1996, 0)

        # Create a dashed vertical line and its corresponding year label for the breaking point.
        v_line_1996 = DashedLine(point_1996, x_point_1996, color=GREEN, stroke_width=6)
        label_1996 = Text("1996", color=GREEN, weight=BOLD, font_size=30).next_to(v_line_1996, UP, buff=0.3)

        # Define the data range for the increase period for highlighting the line segment.
        start_idx = years_to_2025.index(1996)
        end_idx = years_to_2025.index(2025)
        increase_years = years_to_2025[start_idx:end_idx + 1]
        increase_counts = counts_to_2025[start_idx:end_idx + 1]

        # Create a thicker, green line to highlight the period of increase on the chart.
        increase_line = axes.plot_line_graph(
            x_values=increase_years,
            y_values=increase_counts,
            line_color=GREEN,
            stroke_width=16
        )["line_graph"]

        # Animate the display of the breaking point's highlight
        self.play(
            Create(v_line_1996),
            FadeIn(label_1996),
            run_time=1.0
            )

        # Animate the appearance of the thick green increase line.
        self.play(
            FadeIn(increase_line),
            run_time=1.0
            )
        self.wait(1.0)

        # Creation of the scene text - first part.
        text_1 = Text("Zlom nastává v roce 1996", font_size=42)
        text_2 = Text("a populace začíná narůstat.", font_size=42)

        summary_text = VGroup(text_1, text_2).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text.to_corner(UP + RIGHT, buff=0.8)

        self.play(Write(summary_text))
        self.wait(2.0)

        # Creation of the scene text - second part.
        text_3 = Text("V následujících letech se", font_size=42)
        text_4 = Text("populace plynule zvětšuje.", font_size=42)

        summary_text_2 = VGroup(text_3, text_4).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_2.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text, summary_text_2))
        self.wait(3.5)

        # preparation for the scene transition.
        self.play(FadeOut(increase_line,
                          label_1996,
                          v_line_1996,
                          summary_text,
                          )
        )
        self.wait(0.5)

# Sixth Scene - Closer Look

In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph_V

class HorsePopulationGraph_V(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION - FULL GRAPH
        # Create and retrieve the axes and historical graph line using a helper function.
        axes, graph_line = create_horse_graph_elements()

        # Visualize the static chart.
        self.add(axes, graph_line)

        # AXES AND GRAPH PREPARATION - RECENT GRAPH
        axes_recent, graph_line_recent = create_horse_graph_elements(
            x_step=2,
            y_step=25000,
            x_start=2003,
            y_start=0,
            x_end=2025,
            y_end=150000,
            x_values=years_closer_look,
            y_values=counts_closer_look,
            line_color=GREEN,
            stroke_width=10
        )

        # Animate the transition to the closer look.
        self.play(
            Transform(axes, axes_recent),
            Transform(graph_line, graph_line_recent),
            run_time=3.0
        )
        self.wait(1)

        # Creation of the scene text - first part.
        text_1 = Text("Při pohledu na období od roku 2003", font_size=42)
        text_2 = Text("je rostoucí trend zřetelný.", font_size=42)

        summary_text = VGroup(text_1, text_2).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text.to_corner(UP + LEFT, buff=0.8)
        summary_text.shift(RIGHT * 1.2)

        self.play(Write(summary_text))
        self.wait(2.0)

        # Creation of the scene text - second part.
        text_3 = Text("Každý rok se počet koní zvýšil", font_size=42)
        text_4 = Text("v průměru o 3 tis.", font_size=42)

        summary_text_2 = VGroup(text_3, text_4).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_2.to_corner(UP + LEFT, buff=0.8)
        summary_text_2.shift(RIGHT * 1.2)

        self.play(Transform(summary_text, summary_text_2))
        self.wait(3.5)

        # Creation of the scene text - third part.
        text_5 = Text("To představuje průměrný nárust", font_size=42)
        text_6 = Text("o 4.3 % každý rok.", font_size=42)

        summary_text_3 = VGroup(text_5, text_6).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )

        summary_text_3.to_corner(UP + LEFT, buff=0.8)
        summary_text_3.shift(RIGHT * 1.2)

        self.play(Transform(summary_text, summary_text_3))
        self.wait(3.5)

        # Highlight the data point for the year 2025 on the recent graph.
        y_count_2025 = counts_closer_look[years_closer_look.index(2025)]
        point_2025 = axes_recent.c2p(2025, y_count_2025)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_2025 = axes_recent.c2p(2025, 0)

        # Create the highlight line and label for 2025.
        v_line_2025 = DashedLine(point_2025, x_point_2025, color=GREEN, stroke_width=6)
        label_2025 = Text("2025", color=GREEN, weight=BOLD, font_size=30).next_to(v_line_2025, UP, buff=0.3)

        # Animate the display of the 2025 highlight.
        self.play(
            Create(v_line_2025),
            FadeIn(label_2025),
            run_time=1.0
        )

        # Creation of the scene text - fourth part.
        text_7 = Text("V roce 2025 dosáhl počet koní", font_size=42)
        text_8 = Text("hodnoty 110 tis.", font_size=42)

        summary_text_4 = VGroup(text_7, text_8).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_4.to_corner(UP + LEFT, buff=0.8)
        summary_text_4.shift(RIGHT * 1.2)

        self.play(Transform(summary_text, summary_text_4))
        self.wait(3.5)

        # Creation of the scene text - fifth part.
        text_9 = Text("Jak se bude populace koní", font_size=42)
        text_10 = Text("vyvíjet dál?", font_size=42)

        summary_text_5 = VGroup(text_9, text_10).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_5.to_corner(UP + LEFT, buff=0.8)
        summary_text_5.shift(RIGHT * 1.2)

        self.play(Transform(summary_text, summary_text_5))
        self.wait(3.5)

# Seventh Scene - A Glimpse Into the Future I

In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph_VI

class HorsePopulationGraph_VI(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION - RECENT GRAPH
        axes_recent, graph_line_recent = create_horse_graph_elements(
            x_step=2,
            y_step=25000,
            x_start=2003,
            y_start=0,
            x_end=2025,
            y_end=150000,
            x_values=years_closer_look,
            y_values=counts_closer_look,
            line_color=GREEN,
            stroke_width=10
        )

        # Highlight the data point for the year 2025 on the recent graph.
        y_count_2025 = counts_closer_look[years_closer_look.index(2025)]
        point_2025 = axes_recent.c2p(2025, y_count_2025)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_2025 = axes_recent.c2p(2025, 0)

        # Create the highlight line and label for 2025.
        v_line_2025 = DashedLine(point_2025, x_point_2025, color=GREEN, stroke_width=6)
        label_2025 = Text("2025", color=GREEN, weight=BOLD, font_size=30).next_to(v_line_2025, UP, buff=0.3)

        # Creation of the scene text - closer look.
        text_1 = Text("Jak se bude populace koní", font_size=42)
        text_2 = Text("vyvíjet dál?", font_size=42)

        summary_text_1 = VGroup(text_1, text_2).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_1.to_corner(UP + LEFT, buff=0.8)
        summary_text_1.shift(RIGHT * 1.2)

        # Visualize the initial static chart, text, and highlight for 2025.
        self.add(
            axes_recent,
            graph_line_recent,
            summary_text_1,
            label_2025,
            v_line_2025
        )

        # AXES AND GRAPH PREPARATION - FUTURE GRAPH
        axes_future, graph_line_future = create_horse_graph_elements(
            x_step=10,
            y_step=50000,
            x_start=2003,
            y_start=0,
            x_end=2050,
            y_end=250000,
            x_values=years_closer_look,
            y_values=counts_closer_look,
            line_color=GREEN,
            stroke_width=10
        )

        # Highlight the data point for the year 2025 on the future graph.
        point_future_2025 = axes_future.c2p(2025, y_count_2025)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_future_2025 = axes_future.c2p(2025, 0)

        # Create the highlight line and label for 2025.
        v_line_future_2025 = DashedLine(point_future_2025, x_point_future_2025, color=GREEN, stroke_width=6)
        label_future_2025 = Text("2025", color=GREEN, weight=BOLD, font_size=30).next_to(v_line_future_2025, UP, buff=0.3)

        # Animate the transition to the future look.
        self.play(
            Transform(axes_recent, axes_future),
            Transform(graph_line_recent, graph_line_future),
            Transform(label_2025, label_future_2025),
            Transform(v_line_2025, v_line_future_2025),
            run_time=3.0
        )
        self.wait(1.0)

        # Creation of the second scene text - future chart
        text_3 = Text("Bude růst stejným tempem?", font_size=42)
        text_3.to_corner(UP + LEFT, buff=0.8)
        text_3.shift(RIGHT * 1.2)

        self.play(Transform(summary_text_1, text_3))
        self.wait(2.5)

        # Creation of the third scene text - future chart
        text_4 = Text("Budoucnost sice nelze předvídat,", font_size=42)
        text_5 = Text("ale analýza dat nám umožňuje", font_size=42)
        text_6 = Text("odhadnout další vývoj.", font_size=42)

        summary_text_2 = VGroup(text_4, text_5, text_6).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_2.to_corner(UP + LEFT, buff=0.8)
        summary_text_2.shift(RIGHT * 1.2)

        self.play(Transform(summary_text_1, summary_text_2))
        self.wait(3.5)

        # Creation of the fourth scene text - future chart
        text_7 = Text("Podívejme se na předpověď", font_size=42)
        text_8 = Text("až do roku 2050.", font_size=42)

        summary_text_3 = VGroup(text_7, text_8).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_3.to_corner(UP + LEFT, buff=0.8)
        summary_text_3.shift(RIGHT * 1.2)

        # Highlight the data point for the year 2050 on the future graph.
        y_count_future_2050 = counts_upper_2026[years_from_2026.index(2050)]
        point_future_2050 = axes_future.c2p(2050, y_count_future_2050)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_future_2050 = axes_future.c2p(2050, 0)

        # Create the highlight line and label for 2050.
        v_line_future_2050 = DashedLine(point_future_2050, x_point_future_2050, color=YELLOW, stroke_width=6)
        label_future_2050 = Text("2050", color=YELLOW, weight=BOLD, font_size=30).next_to(v_line_future_2050, UP, buff=0.3)

        # Animate the creation of future point.
        self.play(
            Transform(summary_text_1, summary_text_3),
            Create(v_line_future_2050),
            FadeIn(label_future_2050),
            )

        self.wait(3.5)

        # Preparation for the next scene.
        self.play(FadeOut(summary_text_1))
        self.wait(0.5)

# Eight Scene - A Glimpse Into the Future II

In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph_VII

class HorsePopulationGraph_VII(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION - FUTURE GRAPH
        axes_future, graph_line_future = create_horse_graph_elements(
            x_step=10,
            y_step=50000,
            x_start=2003,
            y_start=0,
            x_end=2050,
            y_end=250000,
            x_values=years_closer_look,
            y_values=counts_closer_look,
            line_color=GREEN,
            stroke_width=10
        )

        # Highlight the data point for the year 2025 on the future graph.
        y_count_2025 = counts_closer_look[years_closer_look.index(2025)]
        point_future_2025 = axes_future.c2p(2025, y_count_2025)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_future_2025 = axes_future.c2p(2025, 0)

        # Create the highlight line and label for 2025.
        v_line_future_2025 = DashedLine(point_future_2025, x_point_future_2025, color=GREEN, stroke_width=6)
        label_future_2025 = Text("2025", color=GREEN, weight=BOLD, font_size=30).next_to(v_line_future_2025, UP, buff=0.3)

        # Highlight the data point for the year 2050 on the future graph.
        y_count_future_2050 = counts_upper_2026[years_from_2026.index(2050)]
        point_future_2050 = axes_future.c2p(2050, y_count_future_2050)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_future_2050 = axes_future.c2p(2050, 0)

        # Create the highlight line and label for 2050.
        v_line_future_2050 = DashedLine(point_future_2050, x_point_future_2050, color=YELLOW, stroke_width=6)
        label_future_2050 = Text("2050", color=YELLOW, weight=BOLD, font_size=30).next_to(v_line_future_2050, UP, buff=0.3)

        # Visualize the initial static chart, text, and highlight for 2025 and 2050.
        self.add(
            axes_future,
            graph_line_future,
            label_future_2025,
            v_line_future_2025,
            label_future_2050,
            v_line_future_2050,
        )

        # REPARATION FOR FUTURE PREDICTION AND UNCERTAINTY INTERVALS
        # Prepare the future prediction line from 2025 onwards.
        future_prediction_line = axes_future.plot_line_graph(
            x_values=years_from_2025,
            y_values=counts_from_2025,
            line_color=YELLOW,
            stroke_width=10
        )["line_graph"]

        # Convert the continuous prediction line into a dashed Mobject to visually distinguish it.
        dashed_future_line = DashedVMobject(
            future_prediction_line,
            num_dashes=15,
            dashed_ratio=0.6
        )

        # Prepare the lower bound line of the uncertainty interval.
        lower_bound_line = axes_future.plot_line_graph(
            x_values=years_from_2026,
            y_values=counts_lower_2026,
            line_color=YELLOW,
            stroke_opacity=0.2,
            stroke_width=10
        )["line_graph"]

        # Prepare the upper bound line of the uncertainty interval.
        upper_bound_line = axes_future.plot_line_graph(
            x_values=years_from_2026,
            y_values=counts_upper_2026,
            line_color=YELLOW,
            stroke_opacity=0.2,
            stroke_width=10
        )["line_graph"]

        # Prepare the filled area for the uncertainty interval.
        upper_points = upper_bound_line.points
        lower_points = lower_bound_line.points
        area_points = list(upper_points) + list(lower_points[::-1])

        # Create the polygon Mobject for the uncertainty area.
        uncertainty_area = Polygon(
            *area_points,
            stroke_width=0,
            fill_color=YELLOW,
            fill_opacity=0.5
        )

        # Animate the creation of the dashed future prediction line.
        self.play(
            Create(dashed_future_line),
            run_time=2.0
        )

        # Animate the appearance (fade-in) of the uncertainty interval area and its boundary lines.
        self.play(
            FadeIn(
                uncertainty_area,
                upper_bound_line,
                lower_bound_line),
            run_time=2.0
        )

        self.wait(1.0)

        # Creation of the first scene text - prediction chart
        text_1 = Text("Model ukazuje pravděpodobný", font_size=40)
        text_2 = Text("směr vývoje populace", font_size=40)

        summary_text_1 = VGroup(text_1, text_2).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_1.to_corner(UP + LEFT, buff=0.8)
        summary_text_1.shift(RIGHT * 1.2)

        self.play(Write(summary_text_1))
        self.wait(1.0)

        # Highlight the main prediction line with a passing flash.
        flash_dashed_future_line = dashed_future_line.copy().set_color(GREEN)

        self.play(
            ShowPassingFlash(flash_dashed_future_line, run_time=2.0, time_width=1.0),
        )

        self.wait(0.5)

        # Creation of the second scene text - prediction chart
        text_3 = Text("a predikční interval, znázorňující", font_size=40)
        text_4 = Text("pravděpodobný rozsah hodnot.", font_size=40)

        summary_text_2 = VGroup(text_3, text_4).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_2.to_corner(UP + LEFT, buff=0.8)
        summary_text_2.shift(RIGHT * 1.2)

        self.play(Transform(summary_text_1, summary_text_2))
        self.wait(1.0)

        # Highlight the uncertainty area by temporarily increasing its opacity.
        self.play(
            uncertainty_area.animate.set_fill(opacity=0.8),
            rate_func=there_and_back,
            run_time=2.0
        )
        self.wait(0.5)

        # Creation of the third scene text - prediction chart.
        text_5 = Text("Při současném trendu růstu", font_size=40)
        text_6 = Text("může populace v roce 2050", font_size=40)
        text_7 = Text("dosáhnout hodnoty 175 tis.", font_size=40)

        summary_text_3 = VGroup(text_5, text_6, text_7).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_3.to_corner(UP + LEFT, buff=0.8)
        summary_text_3.shift(RIGHT * 1.2)

        self.play(Transform(summary_text_1, summary_text_3))
        self.wait(0.5)

        # Highlight the point of the prediction for 2050 on the graph with a circle.
        y_count_prediction_2050 = counts_from_2025[years_from_2025.index(2050)]
        point_prediction_2050 = axes_future.c2p(2050, y_count_prediction_2050)
        circle_prediction_2050 = Circle(radius=0.35, color=YELLOW, stroke_width=6).move_to(point_prediction_2050)

        # Animate circle 2050
        self.play(Create(circle_prediction_2050))
        self.wait(3.0)

        # Fadeout circle 2050
        self.play(FadeOut(circle_prediction_2050))

        # Creation of the fourth scene text - prediction chart.
        text_8 = Text("Předpověď se pohybuje v intervalu", font_size=40)
        text_9 = Text("138 až 212 tis. koní.", font_size=40)

        summary_text_4 = VGroup(text_8, text_9).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_4.to_corner(UP + LEFT, buff=0.8)
        summary_text_4.shift(RIGHT * 1.2)

        self.play(Transform(summary_text_1, summary_text_4))
        self.wait(3.5)

        # Creation of the fifth scene text - prediction chart.
        text_10 = Text("I když je interval nejistoty široký,", font_size=40)
        text_11 = Text("hlavní trend zůstává zřejmý:", font_size=40)
        text_12 = Text("tím je růst.", weight=BOLD, font_size=40)

        summary_text_5 = VGroup(text_10, text_11, text_12).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_5.to_corner(UP + LEFT, buff=0.8)
        summary_text_5.shift(RIGHT * 1.2)

        self.play(Transform(summary_text_1, summary_text_5))
        self.wait(3.5)

        self.play(FadeOut(summary_text_1))
        self.wait(0.5)

# Ninth Scene - A Glimpse Into the Future III

In [ ]:
%%manim -qh -v WARNING HorsePopulationGraph_VIII

class HorsePopulationGraph_VIII(Scene):
    def construct(self):
        # AXES AND GRAPH PREPARATION - FUTURE GRAPH
        axes_future, graph_line_future = create_horse_graph_elements(
            x_step=10,
            y_step=50000,
            x_start=2003,
            y_start=0,
            x_end=2050,
            y_end=250000,
            x_values=years_closer_look,
            y_values=counts_closer_look,
            line_color=GREEN,
            stroke_width=10
        )

        # Highlight the data point for the year 2025 on the future graph.
        y_count_2025 = counts_closer_look[years_closer_look.index(2025)]
        point_future_2025 = axes_future.c2p(2025, y_count_2025)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_future_2025 = axes_future.c2p(2025, 0)

        # Create the highlight line and label for 2025.
        v_line_future_2025 = DashedLine(point_future_2025, x_point_future_2025, color=GREEN, stroke_width=6)
        label_future_2025 = Text("2025", color=GREEN, weight=BOLD, font_size=30).next_to(v_line_future_2025, UP, buff=0.3)

        # Highlight the data point for the year 2050 on the future graph.
        y_count_future_2050 = counts_upper_2026[years_from_2026.index(2050)]
        point_future_2050 = axes_future.c2p(2050, y_count_future_2050)

        # Define the point on the x-axis for the base of the vertical highlight line.
        x_point_future_2050 = axes_future.c2p(2050, 0)

        # Create the highlight line and label for 2050.
        v_line_future_2050 = DashedLine(point_future_2050, x_point_future_2050, color=YELLOW, stroke_width=6)
        label_future_2050 = Text("2050", color=YELLOW, weight=BOLD, font_size=30).next_to(v_line_future_2050, UP, buff=0.3)

        # Prepare the future prediction line (median forecast) from 2025 onwards.
        future_prediction_line = axes_future.plot_line_graph(
            x_values=years_from_2025,
            y_values=counts_from_2025,
            line_color=YELLOW,
            stroke_width=10
        )["line_graph"]

        # Convert the continuous prediction line into a dashed Mobject to visually distinguish it.
        dashed_future_line = DashedVMobject(
            future_prediction_line,
            num_dashes=15,
            dashed_ratio=0.6
        )

        # Prepare the lower bound line of the uncertainty interval.
        lower_bound_line = axes_future.plot_line_graph(
            x_values=years_from_2026,
            y_values=counts_lower_2026,
            line_color=YELLOW,
            stroke_opacity=0.2,
            stroke_width=10
        )["line_graph"]

        # Prepare the upper bound line of the uncertainty interval.
        upper_bound_line = axes_future.plot_line_graph(
            x_values=years_from_2026,
            y_values=counts_upper_2026,
            line_color=YELLOW,
            stroke_opacity=0.2,
            stroke_width=10
        )["line_graph"]

        # Prepare the filled area for the uncertainty interval.
        upper_points = upper_bound_line.points
        lower_points = lower_bound_line.points
        area_points = list(upper_points) + list(lower_points[::-1])

        # Create the polygon Mobject for the uncertainty area.
        uncertainty_area = Polygon(
            *area_points,
            stroke_width=0,
            fill_color=YELLOW,
            fill_opacity=0.5
        )

        # Visualize the initial static chart, prediction line, uncertainty interval and highlight for 2025 and 2050.
        self.add(
            axes_future,
            graph_line_future,
            label_future_2025,
            v_line_future_2025,
            label_future_2050,
            v_line_future_2050,
            dashed_future_line,
            uncertainty_area,
            lower_bound_line,
            upper_bound_line,
        )

        # PREPARATION FOR FULL GRAPH WITH PREDICTION
        # AXES AND GRAPH PREPARATION
        axes_full, graph_line_full = create_horse_graph_elements(
            x_step=20,
            y_step=100000,
            x_start=1920,
            y_start=0,
            x_end=2060,
            y_end=600000,
        )

        # Highlight the data point for the year 2025 on the full graph.
        point_full_2025 = axes_full.c2p(2025, y_count_2025)
        x_point_full_2025 = axes_full.c2p(2025, 0)
        v_line_full_2025 = DashedLine(point_full_2025, x_point_full_2025, color=BLUE, stroke_width=6)
        label_full_2025 = Text("2025", color=BLUE, weight=BOLD, font_size=30).next_to(v_line_full_2025, UP, buff=0.3)

        # Highlight the data point for the year 2050 on the full graph.
        point_full_2050 = axes_full.c2p(2050, y_count_future_2050)
        x_point_full_2050 = axes_full.c2p(2050, 0)
        v_line_full_2050 = DashedLine(point_full_2050, x_point_full_2050, color=YELLOW, stroke_width=6)
        label_full_2050 = Text("2050", color=YELLOW, weight=BOLD, font_size=30).next_to(v_line_full_2050, UP, buff=0.3)

        # Prepare the future prediction line (median forecast) from 2025 onwards on full graph.
        prediction_line_full = axes_full.plot_line_graph(
            x_values=years_from_2025,
            y_values=counts_from_2025,
            line_color=YELLOW,
            stroke_width=10
        )["line_graph"]

        # Convert the continuous prediction line into a dashed Mobject to visually distinguish it on full graph.
        dashed_future_line_full = DashedVMobject(
            prediction_line_full,
            num_dashes=15,
            dashed_ratio=0.6
        )

        # Prepare the lower bound line of the uncertainty interval on full graph.
        lower_bound_line_full = axes_full.plot_line_graph(
            x_values=years_from_2026,
            y_values=counts_lower_2026,
            line_color=YELLOW,
            stroke_opacity=0.2,
            stroke_width=10
        )["line_graph"]

        # Prepare the upper bound line of the uncertainty interval on full graph.
        upper_bound_line_full = axes_full.plot_line_graph(
            x_values=years_from_2026,
            y_values=counts_upper_2026,
            line_color=YELLOW,
            stroke_opacity=0.2,
            stroke_width=10
        )["line_graph"]

        # Prepare the filled area for the uncertainty interval on full groph.
        upper_points_full = upper_bound_line_full.points
        lower_points_full = lower_bound_line_full.points
        area_points_full = list(upper_points_full) + list(lower_points_full[::-1])

        # Create the polygon Mobject for the uncertainty area on full chart.
        uncertainty_area_full = Polygon(
            *area_points_full,
            stroke_width=0,
            fill_color=YELLOW,
            fill_opacity=0.5
        )

        # Transformation of the entire scene.
        self.play(
            Transform(axes_future, axes_full),
            Transform(graph_line_future, graph_line_full),
            Transform(label_future_2025, label_full_2025),
            Transform(v_line_future_2025, v_line_full_2025),
            Transform(label_future_2050, label_full_2050),
            Transform(v_line_future_2050, v_line_full_2050),
            Transform(dashed_future_line, dashed_future_line_full),
            Transform(lower_bound_line, lower_bound_line_full),
            Transform(upper_bound_line, upper_bound_line_full),
            Transform(uncertainty_area, uncertainty_area_full),
            run_time=3.0
        )
        self.wait(1.0)

        # Creation of the first scene text.
        text_1 = Text("Populace koní na našem území", font_size=40)
        text_2 = Text("prošla obdobím velkého pádu", font_size=40)
        text_3 = Text("a opětovného vzestupu.", font_size=40)

        summary_text_1 = VGroup(text_1, text_2, text_3).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_1.to_corner(UP + RIGHT, buff=0.8)

        self.play(Write(summary_text_1))
        self.wait(2.0)

        # Creation of the second scene text.
        text_4 = Text("Tento vývoj odráží proměnu", font_size=40)
        text_5 = Text("role koně v naší společnosti.", font_size=40)

        summary_text_2 = VGroup(text_4, text_5).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_2.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text_1, summary_text_2))
        self.wait(3.0)

        # Creation of the third scene text.
        text_6 = Text("Koně měli klíčovou roli", font_size=40)
        text_7 = Text("v zemědělství, dopravě", font_size=40)
        text_8 = Text("a armádě.", font_size=40)

        summary_text_3 = VGroup(text_6, text_7, text_8).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_3.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text_1, summary_text_3))
        self.wait(3.5)

        # Creation of the fourth scene text.
        text_9 = Text("Kvůli tomu byla početnost", font_size=40)
        text_10 = Text("koní naprosto zásadní.", font_size=40)

        summary_text_4 = VGroup(text_9, text_10).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_4.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text_1, summary_text_4))
        self.wait(3.0)

        # Creation of the fifth scene text.
        text_11 = Text("Nástup mechanizace, ztráta", font_size=40)
        text_12 = Text("významu pro armádu", font_size=40)
        text_13 = Text("a změna režimu", font_size=40)
        text_14 = Text("způsobily strmý pád.", font_size=40)

        summary_text_5 = VGroup(text_11, text_12, text_13, text_14).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_5.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text_1, summary_text_5))
        self.wait(3.5)

        # Creation of the sixth scene text.
        text_15 = Text("Koně nevymizeli, naopak", font_size=40)
        text_16 = Text("našli novou roli ve sportu", font_size=40)
        text_17 = Text("a rekreaci.", font_size=40)

        summary_text_6 = VGroup(text_15, text_16, text_17).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_6.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text_1, summary_text_6))
        self.wait(3.5)

        # Creation of the seventh scene text.
        text_18 = Text("Pokles se zastavil a populace", font_size=40)
        text_19 = Text("začala opět narůstat.", font_size=40)

        summary_text_7 = VGroup(text_18, text_19).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_7.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text_1, summary_text_7))
        self.wait(3.0)

        # Creation of the eight scene text.
        text_21 = Text("Příběh koní je příběhem", font_size=40)
        text_22 = Text("pádu a vzestupu.", font_size=40)
        text_23 = Text("Jeho další kapitola", font_size=40)
        text_24 = Text("se právě píše.", font_size=40)

        summary_text_8 = VGroup(text_21, text_22, text_23, text_24).arrange(
            DOWN,
            aligned_edge=LEFT,
            buff=0.3
        )
        summary_text_8.to_corner(UP + RIGHT, buff=0.8)

        self.play(Transform(summary_text_1, summary_text_8))
        self.wait(3.5)

        # Preparing for the transition to the conclusion.
        self.play(FadeOut(*self.mobjects))
        self.wait(0.7)

# Tenth Scene - Conclusion

In [ ]:
%%manim -qh -v WARNING HorsePopulationConclusion

class HorsePopulationConclusion(Scene):
    def construct(self):
        # text
        text_1 = Text("Vytvořeno pomocí Manim", font_size=40, slant=ITALIC)

        # text positioning
        text_1.shift(UP * 2)

        # Manim icon
        manim_icon = ImageMobject(MANIM_ICON_PATH)

        # arrange icon
        manim_icon.next_to(text_1, DOWN)

        # text 2
        text_2 = Text("By David Němeček", font_size=40, slant=ITALIC)
        text_2.next_to(manim_icon, DOWN)

        # animations - text
        self.play(
            FadeIn(text_1),
            FadeIn(manim_icon),
            FadeIn(text_2),
            run_time=2.0
            )
        self.wait(2.5)

        self.play(FadeOut(*self.mobjects))
        self.play(Wait(0.5))